# Volume 3, Chapter 1: Counting and Paradoxes (Python companion)

Exercises first, solutions at the end. Every paradox of the chapter is simulated, with an error bar: a Monte Carlo estimate is worthless without one. We also find a real collision of a (truncated) cryptographic hash with the birthday attack, verify the ballot theorem exhaustively, estimate $\pi$ with needles and catch Lazzarini, and play the secretary problem.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools
import math
import random
import hashlib
import time
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2026)

def estimate(successes, n):
    """Proportion with a 95% confidence interval (normal approximation, justified in Chapter 6)."""
    p = successes / n
    h = 1.96 * math.sqrt(p * (1 - p) / n)
    return p, h

## Exercises

**P1.1 (★) Paradoxes by simulation.** Simulate $10^6$ games of Monty Hall with the knowing host of Proposition 1.9 and with an ignorant host who opens door 2 or 3 at random (discard the games where he reveals the car). Estimate the probability that switching wins in each case, with a 95% confidence interval. Do the same for the two-children problem with a boy born on a Tuesday (Exercise 1.2) and for the medical test of Example 1.6.

In [ ]:
# Your code here


**P1.2 (★★) The birthday attack.** (a) Compute the exact probability $1-q_n$ of Proposition 1.8 for $N=365$ and find the smallest $n$ with $1-q_n\ge\frac12$ and $\ge0.99$. (b) Throw balls into $N$ boxes until the first collision and estimate the mean waiting time for $N=10^6$; compare with $\sqrt{\pi N/2}$. (c) Truncate SHA-256 to its first $b=32$ bits and find two different strings with the same truncated hash. How many hashes did it take, compared with $2^{b/2}$ and with $2^b$?

In [ ]:
# Your code here


**P1.3 (★) Bertrand's chords.** Draw $10^5$ chords by each of the three methods of Proposition 1.10, estimate the probability of a chord longer than $\sqrt3$, and plot the three histograms of chord lengths. Which method gives the uniform distribution of lengths?

In [ ]:
# Your code here


**P1.4 (★) The ballot theorem, exhaustively.** For all $1\le b<a\le 9$, enumerate the $\binom{a+b}a$ orders of the ballots and count those where A is strictly ahead throughout. Check Theorem 1.12 exactly (with fractions), and the Catalan numbers of Exercise 1.7.

In [ ]:
# Your code here


**P1.5 (★★) Buffon's needle and Lazzarini.** Throw $n$ needles with $\ell/d=\frac56$ for $n=10^2,\dots,10^7$, estimate $\pi$, and plot the error against $n$ on log-log axes. What slope do you see? Then compute exactly, with the binomial law, the probability that $3408$ honest throws give an estimate within $10^{-6}$ of $\pi$, as Lazzarini claimed.

In [ ]:
# Your code here


**P1.6 (★★) The secretary problem.** For $n=100$, compute $p_r$ of Exercise 1.8 for every $r$ and find the best $r$. Simulate the threshold strategy with that $r$ and compare. How does the best $r/n$ and the best $p_r$ behave as $n$ grows?

In [ ]:
# Your code here


**P1.7 (★★) Pairwise independence from few bits.** With $k=4$ fair bits, build the $15$ parity events of Exercise 1.6 and check exactly, over the $16$ equally likely outcomes, that they are pairwise independent but not $3$-wise independent. Then use them as a cheap hash: map $15$ keys to bits and check that any two keys collide (same bit) with probability exactly $\frac12$.

In [ ]:
# Your code here


---
## Solutions

### P1.1

In [ ]:
n = 10**6
car = rng.integers(1, 4, n)                       # the player always picks door 1
# knowing host: if car is 1 he opens 2 or 3 at random, otherwise the other goat door
opened = np.where(car == 1, rng.integers(2, 4, n), np.where(car == 2, 3, 2))
switch_door = 5 - opened                         # the remaining closed door among 2, 3
p, h = estimate(np.sum(switch_door == car), n)
print(f'knowing host:  switching wins {p:.4f} +- {h:.4f}   (theory 2/3 = {2/3:.4f})')
# ignorant host: opens 2 or 3 at random; keep only the games where a goat is revealed
opened = rng.integers(2, 4, n)
keep = opened != car
switch_door = 5 - opened
p, h = estimate(np.sum((switch_door == car) & keep), np.sum(keep))
print(f'ignorant host: switching wins {p:.4f} +- {h:.4f}   (theory 1/2), {np.sum(keep)} games kept')

sex = rng.integers(0, 2, (n, 2))                 # 1 = boy
day = rng.integers(0, 7, (n, 2))                 # 2 = Tuesday, say
tuesday_boy = ((sex == 1) & (day == 2)).any(axis=1)
both = (sex == 1).all(axis=1)
p, h = estimate(np.sum(both & tuesday_boy), np.sum(tuesday_boy))
print(f'two children, a Tuesday boy: P(two boys) = {p:.4f} +- {h:.4f}   (theory 13/27 = {13/27:.4f})')

sick = rng.random(n) < 0.01
positive = np.where(sick, rng.random(n) < 0.99, rng.random(n) < 0.05)
p, h = estimate(np.sum(sick & positive), np.sum(positive))
print(f'medical test: P(sick | positive) = {p:.4f} +- {h:.4f}   (theory 1/6 = {1/6:.4f})')

All four estimates land within their error bars of the theory. The two Monty Hall runs differ only in the host's rule, and the answers differ ($\frac23$ against $\frac12$): the simulation has to model the *process* that produced the observation, which is exactly the lesson of Section 1.4. Note the widths of the intervals: about $\pm0.001$ with $10^6$ games, and four times wider with $16$ times fewer. Chapter 6 explains this $1/\sqrt n$ law.

### P1.2

In [ ]:
def p_collision(n, N=365):
    q = 1.0
    for k in range(n):
        q *= (N - k) / N
    return 1 - q
half = next(n for n in range(1, 400) if p_collision(n) >= 0.5)
nine9 = next(n for n in range(1, 400) if p_collision(n) >= 0.99)
print('N = 365: 1/2 reached at n =', half, f'({p_collision(half):.4f}); 0.99 at n =', nine9)

def first_collision(N):
    seen = set()
    while True:
        x = random.randrange(N)
        if x in seen: return len(seen) + 1
        seen.add(x)
random.seed(1)
N = 10**6
waits = [first_collision(N) for _ in range(2000)]
p_mean, sd = np.mean(waits), np.std(waits)
print(f'N = 10^6: mean wait {p_mean:.0f} +- {1.96*sd/math.sqrt(len(waits)):.0f}, sqrt(pi N / 2) = {math.sqrt(math.pi*N/2):.0f}')

def h32(s):
    return hashlib.sha256(s.encode()).digest()[:4]
t0 = time.perf_counter()
seen, i = {}, 0
while True:
    s = f'message-{i}'
    d = h32(s)
    if d in seen:
        break
    seen[d] = s; i += 1
print(f'collision after {i+1} hashes ({time.perf_counter()-t0:.1f} s): {seen[d]!r} and {s!r}')
print('   truncated SHA-256 of both:', h32(seen[d]).hex(), h32(s).hex())
print(f'   2^16 = {2**16}, sqrt(pi 2^32 / 2) = {math.sqrt(math.pi*2**32/2):.0f}, 2^32 = {2**32}')

The exact computation confirms $23$ and $57$. The mean waiting time for the first collision is $\sqrt{\pi N/2}\approx1253$ for $N=10^6$ (a classical asymptotic, which goes back to Ramanujan). For a 32-bit hash, a collision appears after a number of hashes of the order of $2^{16}$ (the mean is $\sqrt{\pi2^{32}/2}\approx82\,000$; this run was lucky), instead of the $2^{32}\approx4\times10^9$ one might fear. Real hash functions have $b=256$, so the birthday attack needs about $2^{128}$ hashes: out of reach, but only because $b$ was doubled with this attack in mind.

### P1.3

In [ ]:
m = 10**5
a, b = rng.uniform(0, 2*np.pi, m), rng.uniform(0, 2*np.pi, m)
L1 = 2 * np.abs(np.sin((a - b) / 2))
r = rng.uniform(0, 1, m)
L2 = 2 * np.sqrt(1 - r**2)
# uniform point in the disc: radius sqrt(U) (Chapter 5, change of variables)
r = np.sqrt(rng.uniform(0, 1, m))
L3 = 2 * np.sqrt(1 - r**2)
for name, L, th in [('endpoints', L1, 1/3), ('radius', L2, 1/2), ('midpoint', L3, 1/4)]:
    p, h = estimate(np.sum(L > math.sqrt(3)), m)
    print(f'{name:10s} P(L > sqrt 3) = {p:.4f} +- {h:.4f}   theory {th:.4f}')
fig, ax = plt.subplots(figsize=(7, 3))
for name, L in [('endpoints', L1), ('point on a radius', L2), ('midpoint', L3)]:
    ax.hist(L, bins=60, density=True, histtype='step', lw=1.5, label=name)
ax.axvline(math.sqrt(3), color='grey', ls=':')
ax.set_xlabel('chord length'); ax.legend(); plt.show()

None of the three gives uniformly distributed lengths: the length density is $\frac{1}{\pi\sqrt{1-L^2/4}}$ for endpoints (piling up near the diameter $2$), $\frac{L}{2\sqrt{4-L^2}}$ for the radius method, and $\frac L2$ for midpoints. The midpoint method favours short chords, because most of the disc's area is near its edge. A fourth method, uniform length, would give yet another answer, $1-\frac{\sqrt3}2\approx0.134$.

### P1.4

In [ ]:
def ballot_count(a, b):
    good = 0
    for pos in itertools.combinations(range(a + b), b):     # positions of B's votes
        s, ok = 0, True
        pos = set(pos)
        for k in range(a + b):
            s += -1 if k in pos else 1
            if s <= 0: ok = False; break
        good += ok
    return good
ok = all(Fraction(ballot_count(a, b), math.comb(a + b, a)) == Fraction(a - b, a + b)
         for a in range(2, 10) for b in range(1, a))
print('ballot theorem verified for all 1 <= b < a <= 9:', ok)
print('Dyck paths via ballot (a = n+1, b = n):', [ballot_count(n + 1, n) for n in range(0, 9)])
print('Catalan numbers                       :', [math.comb(2*n, n) // (n + 1) for n in range(0, 9)])

Exact agreement in all $36$ cases, with fractions, so no rounding can hide an error. The second check uses the bijection of Exercise 1.7: counts with $a=n+1$, $b=n$ are Dyck paths, and their numbers $1,1,2,5,14,42,132,429,1430$ are the Catalan numbers, which return in Chapter 3.

### P1.5

In [ ]:
def buffon(n, l_over_d=5/6):
    x = rng.uniform(0, 0.5, n)                 # distance to the nearest line, d = 1
    th = rng.uniform(0, np.pi / 2, n)
    c = np.sum(x <= l_over_d / 2 * np.sin(th))
    return 2 * l_over_d * n / c
ns = [10**k for k in range(2, 8)]
errs = []
for n in ns:
    e = np.mean([abs(buffon(n) - np.pi) for _ in range(20)])
    errs.append(e)
    print(f'n = {n:>9,}: mean |error| over 20 runs = {e:.2e}')
slope = np.polyfit(np.log10(ns), np.log10(errs), 1)[0]
fig, ax = plt.subplots(figsize=(6, 3))
ax.loglog(ns, errs, 'o-', label=f'observed (slope {slope:.2f})')
ax.loglog(ns, [errs[0] * (ns[0] / n) ** 0.5 for n in ns], '--', label='slope $-1/2$')
ax.set_xlabel('number of needles'); ax.set_ylabel('mean |error|'); ax.legend(); plt.show()

# Lazzarini: 3408 throws, l/d = 5/6, crossing probability p = 2l/(pi d) = 5/(3 pi)
n, p = 3408, 5 / (3 * math.pi)
from math import lgamma, log, exp
def pmf(c):
    return exp(lgamma(n + 1) - lgamma(c + 1) - lgamma(n - c + 1) + c * log(p) + (n - c) * log(1 - p))
good = [c for c in range(1, n + 1) if abs(2 * 5 * n / (6 * c) - math.pi) < 1e-6]
print('crossing counts giving pi within 1e-6:', good, ' estimate', 2 * 5 * n / (6 * good[0]))
print(f'probability for an honest experiment: {sum(pmf(c) for c in good):.4f}')
print(f'typical error with 3408 throws: about {np.mean([abs(buffon(3408) - np.pi) for _ in range(2000)]):.3f}')

The slope is $-\frac12$: the error decreases like $1/\sqrt n$, so each extra correct digit costs a hundred times more needles. With $3408$ throws the typical error is about $0.04$. Only one crossing count, $1808$, gives an estimate within $10^{-6}$ of $\pi$, and an honest experiment hits it with probability about $1.4\%$; but Lazzarini's real trick was to choose $n=3408=16\cdot213$ so that $\frac{355}{113}$ was *reachable* (the estimate is $\frac{2\cdot5\cdot3408}{6c}$, which equals $\frac{355}{113}$ only when $c=1808$). Had he reported, say, $3407$ throws, no count would have come within $10^{-4}$.

### P1.6

In [ ]:
def p_r(n, r):
    if r == 1: return 1 / n
    return (r - 1) / n * sum(1 / (k - 1) for k in range(r, n + 1))
n = 100
best = max(range(1, n + 1), key=lambda r: p_r(n, r))
print(f'n = {n}: best r = {best} (n/e = {n/math.e:.1f}), p = {p_r(n, best):.4f}')
def play(n, r):
    vals = rng.permutation(n)
    threshold = vals[:r - 1].max() if r > 1 else -1
    for v in vals[r - 1:]:
        if v > threshold:
            return v == n - 1
    return False
trials = 100_000
wins = sum(play(n, best) for _ in range(trials))
p, h = estimate(wins, trials)
print(f'simulation: {p:.4f} +- {h:.4f}')
for n in [10, 100, 1000, 10000]:
    rs = range(1, n + 1)
    vals = [p_r(n, r) for r in rs] if n <= 1000 else None
    best = max(rs, key=lambda r: p_r(n, r)) if n <= 1000 else round(n / math.e) + 1
    print(f'n = {n:>6}: best r/n = {best/n:.4f}, p = {p_r(n, best):.5f}   (1/e = {1/math.e:.5f})')

The best threshold for $n=100$ rejects the first $37$ candidates and wins $37\%$ of the time, and the simulation agrees. As $n$ grows, both the best fraction $r/n$ and the winning probability tend to $\frac1e\approx0.3679$: remarkably, the chance of picking the very best of a million candidates is still more than a third.

### P1.7

In [ ]:
k = 4
outcomes = list(itertools.product([0, 1], repeat=k))         # 16 equally likely
subsets = [S for r in range(1, k + 1) for S in itertools.combinations(range(k), r)]
event = {S: frozenset(o for o in outcomes if sum(o[i] for i in S) % 2 == 0) for S in subsets}
P = lambda A: Fraction(len(A), len(outcomes))
print(len(subsets), 'events, each of probability', {P(A) for A in event.values()})
pairwise = all(P(event[S] & event[T]) == P(event[S]) * P(event[T]) for S, T in itertools.combinations(subsets, 2))
triples = sum(P(event[S] & event[T] & event[U]) == Fraction(1, 8) for S, T, U in itertools.combinations(subsets, 3))
print('pairwise independent:', pairwise, '  independent triples:', triples, 'of', math.comb(15, 3))
# as a hash: key S gets the bit parity_S(x) for a random seed x of k bits
coll = {(S, T): Fraction(sum((sum(o[i] for i in S) % 2) == (sum(o[i] for i in T) % 2) for o in outcomes), 16)
        for S, T in itertools.combinations(subsets, 2)}
print('collision probability of any two keys:', set(coll.values()))

All $105$ pairs are independent, and $420$ of the $455$ triples are independent as well; the $35$ others are the triples $\{S,T,S\triangle T\}$, exactly the ones the exercise names. So $4$ random bits produce $15$ bits that look independent to any test involving two of them at a time. This is the idea behind pairwise independent hash families, whose collision probability for two given keys is exactly that of a truly random function, at the cost of a few random bits instead of one per key.